In [ ]:
req_pkgs <- c(
  "devtools","ggplot2","dplyr","tidyr","tibble","scales",
  "MASS","numDeriv","matrixStats","purrr","readr","patchwork"
)
need <- setdiff(req_pkgs, rownames(installed.packages()))
if (length(need)) install.packages(need, dependencies = TRUE)
invisible(lapply(req_pkgs, require, character.only = TRUE))

options(warn = 1)
options(error = function() { traceback(2); stop("Stopped on error (see traceback above).", call. = FALSE) })

## --- repo root (works when running from notebooks/) ---
repo_root <- tryCatch(
  normalizePath(system("git rev-parse --show-toplevel", intern = TRUE)),
  error = function(...) normalizePath("..", mustWork = TRUE)
)

to_rm <- intersect(
  c("%||%", ".stopf", "assert_scalar_numeric", "assert_matrix"),
  ls(envir = .GlobalEnv, all.names = TRUE)
)
if (length(to_rm)) rm(list = to_rm, envir = .GlobalEnv)

devtools::load_all(repo_root)
set.seed(12345)
need_funs <- c(
  "qdesn_fit_vb","qdesn_build_design",
  "exal_ldvb_fit","beta_prior",
  "exal_get_ABC"
)

missing <- need_funs[!vapply(need_funs, exists, logical(1), mode = "function", inherits = TRUE)]
if (length(missing)) {
  cat("Missing functions:\n"); print(missing)
  cat("\nObjects matching 'qdesn' in search path:\n")
  print(ls(pattern = "qdesn", all.names = TRUE))
  stop("Required functions missing. load_all() did not expose them.", call. = FALSE)
}

# Optional: if you have L.fn/U.fn in the package, great. If not, we will fallback later.
cat("Has L.fn:", exists("L.fn", mode="function", inherits=TRUE), "\n")
cat("Has U.fn:", exists("U.fn", mode="function", inherits=TRUE), "\n")
data_path <- "/data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_smallW/series_long.csv"

df_long <- readr::read_csv(data_path, show_col_types = FALSE)

stopifnot(all(c("t","p","y") %in% names(df_long)))
df_y <- df_long %>%
  arrange(t, p) %>%
  group_by(t) %>%
  summarise(
    y = dplyr::first(y),
    mu_true = dplyr::first(mu),
    .groups = "drop"
  ) %>%
  arrange(t)

y <- df_y$y
T <- length(y)

cat("T =", T, "\n")
print(head(df_y, 3))
print(tail(df_y, 3))

# quick plot
ggplot(df_y, aes(x = t, y = y)) +
  geom_line() +
  labs(title = "Extracted series y(t)", x = "t", y = "y")
p0 <- 0.05

# gamma bounds: prefer your L.fn/U.fn; else use a safe wide interval
gamma_bounds <- if (exists("L.fn", mode="function", inherits=TRUE) &&
                    exists("U.fn", mode="function", inherits=TRUE)) {
  c(L.fn(p0), U.fn(p0))
} else {
  c(-30, 30)
}

# "non-informative" defaults (practical):
# beta: very weak ridge
beta_prior_obj <- beta_prior("ridge", ridge = list(tau2 = 1e8))

# gamma: very wide Normal
prior_gamma <- list(mu0 = 0, s20 = 1e6)

# sigma: weak IG (proper but close to flat on log-scale over a wide range)
prior_sigma <- list(a = 1e-3, b = 1e-3)

vb_control <- list(
  max_iter = 1000L,
  tol      = 1e-4,
  tol_par  = 1e-4,
  min_iter_elbo = 100L,
  verbose  = TRUE
)

gamma_bounds
fit_qdesn <- qdesn_fit_vb(
  y = y,
  p0 = p0,

  # keep your defaults unless you want smaller for debugging
  D = 3L,
  n = c(150L, 150L, 150L),
  n_tilde = c(150L, 150L),
  m = 90L,
  alpha = 0.15,
  rho = rep(0.95, 3L),
  act_f = "tanh",
  act_k = "identity",
  pi_w = 0.1,
  pi_in = 0.1,
  washout = 100L,
  add_bias = TRUE,

  seed = 12345,

  vb_args = list(
    # if qdesn_fit_vb forwards these to exal_static_LDVB (old) or exal_ldvb_fit (new)
    vb_control = vb_control,
    gamma_bounds = gamma_bounds,
    prior_gamma = prior_gamma,
    prior_sigma = prior_sigma,
    beta_prior_obj = beta_prior_obj
  ),

  fit_readout = TRUE
)

str(fit_qdesn, max.level = 2)

# If your object stores readout fit as $fit (per header comment):
if (!is.null(fit_qdesn$fit)) {
  cat("Readout converged:", fit_qdesn$fit$converged, " iter:", fit_qdesn$fit$iter, "\n")
  cat("gamma_hat:", fit_qdesn$fit$qsiggam$gamma_mean, " sigma_hat:", fit_qdesn$fit$qsiggam$sigma_mean, "\n")
}
desn_args <- list(
  D = 3L,
  n = c(150L, 150L, 150L),
  n_tilde = c(150L, 150L),
  m = 90L,
  alpha = 0.15,
  rho = rep(0.95, 3L),
  act_f = "tanh",
  act_k = "identity",
  pi_w = 0.1,
  pi_in = 0.1,
  washout = 500L,
  add_bias = TRUE,
  seed = 12345,

  # IMPORTANT: we only want the design here
  fit_readout = FALSE
)

des <- qdesn_build_design(y = y, desn_args = desn_args)
X <- des$X
keep_idx <- des$keep_idx

y_fit <- y[keep_idx]

cat("Design dim:", dim(X), "  y_fit length:", length(y_fit), "\n")
stopifnot(nrow(X) == length(y_fit))

fit_readout <- exal_ldvb_fit(
  y = y_fit,
  X = X,
  p0 = p0,
  gamma_bounds = gamma_bounds,
  vb_control = vb_control,
  init = list(),                 # keep default init
  prior_gamma = prior_gamma,
  prior_sigma = prior_sigma,
  beta_prior_obj = beta_prior_obj
)

cat("Converged:", fit_readout$converged, " iter:", fit_readout$iter, " time:", fit_readout$run.time, "sec\n")
cat("gamma_hat:", fit_readout$qsiggam$gamma_mean, " sigma_hat:", fit_readout$qsiggam$sigma_mean, "\n")
vb <- fit_readout

df_tr <- tibble::tibble(
  iter = seq_along(vb$misc$elbo_trace),
  elbo = vb$misc$elbo_trace,
  gamma = vb$misc$gamma_trace,
  sigma = vb$misc$sigma_trace,
  new_term = vb$misc$new_term_trace
)

p1 <- ggplot(df_tr, aes(iter, elbo)) + geom_line() + labs(title="ELBO (per obs)")
p2 <- ggplot(df_tr, aes(iter, gamma)) + geom_line() + labs(title="gamma trace")
p3 <- ggplot(df_tr, aes(iter, sigma)) + geom_line() + labs(title="sigma trace")
p4 <- ggplot(df_tr, aes(iter, pmax(new_term, 1e-16))) +
  geom_line() + scale_y_log10() + labs(title="new_term (log10, floored)")

p1 / (p2 | p3) / p4
beta_hat <- vb$qbeta$m
mu_hat <- as.numeric(X %*% beta_hat)

df_fit <- tibble::tibble(
  t = keep_idx,
  y = y_fit,
  mu_hat = mu_hat,
  mu_true = df_y$mu_true[keep_idx]
)

ggplot(df_fit, aes(t, y)) +
  geom_line() +
  geom_line(aes(y = mu_hat)) +
  geom_line(aes(y = mu_true), linetype = 2) +
  labs(title = "y vs fitted mu_hat vs true mu (dashed)", x = "t", y = "")

library(dplyr)
library(ggplot2)
library(tibble)

# Align to the kept time indices (washout dropped)
keep_idx <- fit_qdesn$meta$keep_idx
t_keep   <- keep_idx
y_keep   <- y[keep_idx]

# fitted p0-quantile (location)
mu_hat <- fit_qdesn$mu_hat
if (is.null(mu_hat)) {
  mu_hat <- as.numeric(fit_qdesn$X %*% fit_qdesn$fit$qbeta$m)
}

# optional: true latent mean/quantile from your CSV (you already built df_y with mu_true)
mu_true_keep <- df_y$mu_true[keep_idx]

df_plot <- tibble(
  t = t_keep,
  y = y_keep,
  mu_hat = mu_hat,
  mu_true = mu_true_keep
)

df_last <- df_plot %>% slice_tail(n = 200)

# Overlay plot on last 1000 kept points
ggplot(df_last, aes(x = t)) +
  geom_line(aes(y = y), alpha = 0.6) +
  geom_line(aes(y = mu_hat), linewidth = 1) +
  geom_line(aes(y = mu_true), linetype = 2, alpha = 0.9) +
  labs(
    title = sprintf("Last 1000 points: y vs fitted p0-quantile (p0=%.2f)", p0),
    x = "t", y = ""
  )

# Quick numeric diagnostics on last 1000
check_loss <- function(u, p) u * (p - as.numeric(u < 0))

u <- df_last$y - df_last$mu_hat
cat("MAE:", mean(abs(u)), "\n")
cat("RMSE:", sqrt(mean(u^2)), "\n")
cat("Check loss (pinball):", mean(check_loss(u, p0)), "\n")
cat("Corr(y, mu_hat):", cor(df_last$y, df_last$mu_hat), "\n")

vb <- fit_readout
L <- gamma_bounds[1]; U <- gamma_bounds[2]
gamma_hat <- vb$qsiggam$gamma_mean
sigma_hat <- vb$qsiggam$sigma_mean

cat("dist to bounds:", min(gamma_hat - L, U - gamma_hat), "\n")  # should not be ~0
cat("sigma_hat:", sigma_hat, "\n")

# ELBO should stabilize; small non-monotone wiggles can happen with delta approx,
# but it should not be exploding or going to -Inf/NA.
el <- vb$misc$elbo_trace %||% vb$misc$elbo
cat("ELBO finite:", all(is.finite(el)), "\n")
cat("ELBO last diffs summary:\n"); print(summary(diff(tail(el, 50))))

# Empirical quantile coverage (training)
mu_hat <- as.numeric(X %*% vb$qbeta$m)
cat("coverage P(y <= mu_hat):", mean(y_fit <= mu_hat), " target:", p0, "\n")
